# LeetCode 565 — Array Nesting bằng Python

Link đề: https://leetcode.com/problems/array-nesting/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Hoán vị (permutation), Chu trình (cycle), Union-Find

Skeleton LeetCode:

```python
class Solution:
    def arrayNesting(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài này không cần cấu trúc tự dựng (chỉ là mảng số nguyên). Mọi lời giải đều được đối chiếu với một "oracle" làm đúng từng chữ theo định nghĩa trong đề (tự dựng tập `s[k]` cho từng `k`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` là một **hoán vị** của `0..n-1`; từ chỉ số `k`, ta liên tục "nhảy" `k -> nums[k] -> nums[nums[k]] -> ...` cho tới khi sắp gặp lại một phần tử đã có, và phải tìm **tập dài nhất** như vậy.
2. Nắm các công cụ Python cần thiết: truy cập lồng `nums[nums[i]]`, `set` và toán tử `in`, mảng cờ `visited = [False] * n`, vòng `while`, phép hoán đổi `a[i], a[j] = a[j], a[i]` (và cái bẫy thứ tự gán), và `time.perf_counter` để đo thời gian.
3. Đi từ **cách thô** (mô phỏng từ mọi `k`, dùng `set`, `O(n²)`) → **đánh dấu đã thăm để mỗi phần tử chỉ đi qua một lần** (`O(n)`) → **đánh dấu ngay trên mảng đầu vào** (`O(1)` bộ nhớ phụ) → **Union-Find** → **hoán đổi kiểu cycle sort** (đếm số lần hoán đổi).
4. Hiểu mấu chốt: một hoán vị tách mảng thành các **chu trình rời nhau**; tập `s[k]` chính là chu trình chứa `k`, nên đáp án = **độ dài chu trình dài nhất**, và mỗi chu trình chỉ cần đi **một lần**.
5. Tránh các lỗi kinh điển: không đánh dấu đã thăm (bị `O(n²)` và quá thời gian), chỉ đánh dấu điểm xuất phát, viết sai phép hoán đổi, đếm thiếu/thừa 1 phần tử, và đệ quy quá sâu với chu trình dài tới `10^5`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n`, trong đó `nums` là một **hoán vị** của các số trong đoạn `[0, n - 1]` (mỗi số từ `0` đến `n-1` xuất hiện **đúng một lần**).

Với mỗi chỉ số `k`, ta xây tập `s[k] = {nums[k], nums[nums[k]], nums[nums[nums[k]]], ...}` theo luật:

- Phần tử đầu tiên của `s[k]` là `nums[k]` (phần tử có chỉ số `k`).
- Phần tử kế tiếp là `nums[nums[k]]`, rồi `nums[nums[nums[k]]]`, và cứ thế.
- Ta **dừng ngay trước khi** một phần tử trùng lặp xuất hiện trong `s[k]`.

Hãy trả về **độ dài lớn nhất** của một tập `s[k]`.

**Ví dụ 1**

- Input: `nums = [5,4,0,3,1,6,2]`
- Output: `4`
- Giải thích: `s[0] = {nums[0], nums[5], nums[6], nums[2]} = {5, 6, 2, 0}`, có 4 phần tử, và không tập nào dài hơn.

**Ví dụ 2**

- Input: `nums = [0,1,2]`
- Output: `1`
- Giải thích: `nums[k] = k` với mọi `k`, nên `s[k] = {k}`, độ dài `1`.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] < nums.length`
- Mọi giá trị trong `nums` là **duy nhất** (tức `nums` là hoán vị).

Chú ý: với `n` lên tới `10^5` thì mọi cách có độ phức tạp `O(n²)` (khoảng `10^10` bước) đều **quá chậm**; ta cần một cách gần tuyến tính.

In [1]:
from typing import List
import random
import time
import sys
from math import gcd

examples = [
    ([5, 4, 0, 3, 1, 6, 2], 4),
    ([0, 1, 2], 1),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected, "| n =", len(nums), "| là hoán vị:", sorted(nums) == list(range(len(nums))))

[5, 4, 0, 3, 1, 6, 2] -> kỳ vọng 4 | n = 7 | là hoán vị: True
[0, 1, 2] -> kỳ vọng 1 | n = 3 | là hoán vị: True


## 3. Kiến thức nền

### 3.1. Mảng là một "bảng tra": `nums[i]` vừa là giá trị, vừa là **chỉ số kế tiếp**

Trong bài này, giá trị `nums[i]` luôn nằm trong `0..n-1`, nên **nó cũng là một chỉ số hợp lệ**. Vì vậy ta được phép viết lồng: `nums[nums[i]]` nghĩa là "lấy giá trị `nums[i]`, rồi coi nó như chỉ số để tra tiếp".

Hãy nghĩ `nums` như một dãy mũi tên: ô `i` có mũi tên trỏ tới ô `nums[i]`.

In [2]:
nums = [5, 4, 0, 3, 1, 6, 2]
i = 0
print("nums[0] =", nums[0])
print("nums[nums[0]] = nums[5] =", nums[nums[0]])
print("nums[nums[nums[0]]] = nums[6] =", nums[nums[nums[0]]])

# Đi theo mũi tên 4 bước từ ô 0
cur = 0
for step in range(4):
    cur = nums[cur]
    print("sau", step + 1, "bước ->", cur)

nums[0] = 5
nums[nums[0]] = nums[5] = 6
nums[nums[nums[0]]] = nums[6] = 2
sau 1 bước -> 5
sau 2 bước -> 6
sau 3 bước -> 2
sau 4 bước -> 0


### 3.2. `set` và toán tử `in`

`set` là tập hợp **không có phần tử trùng**. Hai thao tác ta cần:

- `s.add(x)`: thêm `x` vào tập (nếu đã có thì không đổi).
- `x in s`: kiểm tra `x` có trong tập không, trung bình **`O(1)`**.

Trong đề, "dừng ngay trước khi có phần tử trùng" chính là: lặp cho tới khi `x in s` trở thành đúng. `len(s)` cho biết số phần tử.

In [3]:
s = set()
for x in [5, 6, 2, 0, 5, 6]:
    print("x =", x, "đã có trong s?", x in s)
    s.add(x)
print(s, "-> len =", len(s))

x = 5 đã có trong s? False
x = 6 đã có trong s? False
x = 2 đã có trong s? False
x = 0 đã có trong s? False
x = 5 đã có trong s? True
x = 6 đã có trong s? True
{0, 2, 5, 6} -> len = 4


### 3.3. Mảng cờ `visited`

Để nhớ ô nào đã đi qua, ta dùng một list boolean cùng độ dài với `nums`: `visited = [False] * n`. Khi đi qua ô `i` thì gán `visited[i] = True`. Tra cứu và gán đều `O(1)`, và nhanh hơn `set` vì chỉ là truy cập chỉ số mảng.

Cẩn thận: `[False] * n` là đúng; tuy nhiên `[[False]] * n` (list lồng) mới là cái bẫy dùng chung đối tượng. Ở đây list phẳng nên an toàn.

In [4]:
n = 5
visited = [False] * n
visited[2] = True
visited[4] = True
print(visited)
print("số ô đã thăm:", sum(visited))
print("ô chưa thăm:", [i for i in range(n) if not visited[i]])

[False, False, True, False, True]
số ô đã thăm: 2
ô chưa thăm: [0, 1, 3]


### 3.4. Vòng `while` với điều kiện dừng "gặp lại điểm xuất phát"

Khi ta đi theo mũi tên trong một hoán vị, ta **không bao giờ rơi vào ngõ cụt** và cũng **không bao giờ lặp lại một phần tử tùy ý**: ta chỉ có thể quay lại đúng **điểm xuất phát**. (Lý do sẽ chứng minh ở mục 6 và 10.) Nhờ vậy điều kiện dừng đơn giản là "đã quay về ô bắt đầu".

Ví dụ đếm số bước để vòng về lại `0` với `nums = [5,4,0,3,1,6,2]`.

In [5]:
nums = [5, 4, 0, 3, 1, 6, 2]
start = 0
cur = nums[start]
count = 1                     # đã đi 1 bước (tới nums[start])
while cur != start:
    cur = nums[cur]
    count += 1
print("từ ô", start, "mất", count, "bước để quay về chính nó")

từ ô 0 mất 4 bước để quay về chính nó


### 3.5. Hoán đổi hai phần tử và cái bẫy thứ tự gán

Python hoán đổi bằng gán bộ: `a[i], a[j] = a[j], a[i]`. Vế phải được **tính hết trước**, rồi vế trái được gán **từ trái sang phải**. Chính vì gán từ trái sang phải mà có một cái bẫy khi chỉ số `j` phụ thuộc vào `a[i]`:

```python
a[i], a[a[i]] = a[a[i]], a[i]   # SAI nếu a[i] bị đổi ở lần gán đầu
```

Lần gán đầu làm `a[i]` thay đổi, nên lần gán thứ hai `a[a[i]]` dùng **`a[i]` mới**, ghi sai vị trí. Cách an toàn là lưu chỉ số vào một biến riêng `j = a[i]` trước.

In [6]:
a = [2, 0, 1]
i = 0
a[i], a[a[i]] = a[a[i]], a[i]
print("cách viết bẫy  :", a, "(mất số 0, trùng số 1 -> hỏng hoán vị)")

a = [2, 0, 1]
i = 0
j = a[i]
a[i], a[j] = a[j], a[i]
print("cách an toàn   :", a, "(vẫn là hoán vị)")

cách viết bẫy  : [1, 2, 1] (mất số 0, trùng số 1 -> hỏng hoán vị)
cách an toàn   : [1, 0, 2] (vẫn là hoán vị)


### 3.6. Đo thời gian với `time.perf_counter`

Để thấy tận mắt sự khác nhau giữa `O(n²)` và `O(n)`, ta đo thời gian chạy. `time.perf_counter()` trả về giây dưới dạng số thực; lấy hiệu giữa hai lần gọi. Hàm `timeit_once` dưới đây dùng lại nhiều lần.

In [7]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

res, dt = timeit_once(sum, range(10 ** 6))
print("sum =", res, "| thời gian khoảng", round(dt * 1000, 2), "ms")

sum = 499999500000 | thời gian khoảng 28.26 ms


## 4. Ý tưởng nền tảng / trực giác

Hãy vẽ ví dụ 1 thành các mũi tên `i -> nums[i]`, với `nums = [5,4,0,3,1,6,2]`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| nums[i] | 5 | 4 | 0 | 3 | 1 | 6 | 2 |

Tính tay:

- Từ `0`: `0 -> 5 -> 6 -> 2 -> 0`. Vòng về `0`, gồm `{0, 5, 6, 2}` (4 phần tử).
- Từ `1`: `1 -> 4 -> 1`. Vòng gồm `{1, 4}` (2 phần tử).
- Từ `3`: `3 -> 3`. Vòng gồm `{3}` (1 phần tử).

Quan sát quan trọng:

1. Mỗi ô có **đúng một mũi tên đi ra** (`nums[i]`) và **đúng một mũi tên đi vào** (vì mỗi giá trị xuất hiện đúng 1 lần).
2. Do đó, đi theo mũi tên thì phải **quay lại đúng ô xuất phát**; các ô chia thành các **vòng khép kín rời nhau**.
3. Bắt đầu từ bất kỳ ô nào trong cùng một vòng, ta đi qua đúng cả vòng đó, nên độ dài như nhau: `s[0]`, `s[5]`, `s[6]`, `s[2]` đều có 4 phần tử.

Vậy **đáp án = độ dài của vòng dài nhất**. Hãy kiểm chứng bằng code: in vòng của từng `k`.

In [8]:
def nesting_list(nums, k):
    """Danh sách s[k] theo đúng thứ tự trong đề (dừng ngay trước khi có phần tử trùng)."""
    seen = set()
    order = []
    cur = nums[k]
    while cur not in seen:
        seen.add(cur)
        order.append(cur)
        cur = nums[cur]
    return order

nums = [5, 4, 0, 3, 1, 6, 2]
for k in range(len(nums)):
    sk = nesting_list(nums, k)
    print(f"s[{k}] = {sk}  (độ dài {len(sk)})")

s[0] = [5, 6, 2, 0]  (độ dài 4)
s[1] = [4, 1]  (độ dài 2)
s[2] = [0, 5, 6, 2]  (độ dài 4)
s[3] = [3]  (độ dài 1)
s[4] = [1, 4]  (độ dài 2)
s[5] = [6, 2, 0, 5]  (độ dài 4)
s[6] = [2, 0, 5, 6]  (độ dài 4)


Nhìn kết quả: `s[0], s[2], s[5], s[6]` cùng là các phần tử `{0, 2, 5, 6}` (chỉ khác thứ tự bắt đầu), và `s[1], s[4]` cùng là `{1, 4}`. Mỗi `k` chỉ "rơi" vào vòng của mình, và mọi `k` trong cùng vòng cho cùng độ dài. Điều này gợi ngay một ý tối ưu: **đừng tính lại** cho các `k` cùng vòng.

## 5. Cách 1 — Brute Force: mô phỏng đúng đề cho từng `k`

### Ý tưởng

Làm đúng từng chữ theo đề: với mỗi `k` từ `0` đến `n-1`, dựng `s[k]` bằng một `set` mới (đi theo mũi tên tới khi gặp phần tử đã có), lấy `len` rồi cập nhật đáp án lớn nhất.

### Code

In [9]:
def array_nesting_brute(nums):
    best = 0
    for k in range(len(nums)):
        seen = set()
        cur = nums[k]
        while cur not in seen:
            seen.add(cur)
            cur = nums[cur]
        best = max(best, len(seen))
    return best

for nums, expected in examples:
    got = array_nesting_brute(nums)
    print(nums, "->", got)
    assert got == expected

[5, 4, 0, 3, 1, 6, 2] -> 4
[0, 1, 2] -> 1


### Độ phức tạp và nút thắt

- **Time:** mỗi `k` mất bằng độ dài vòng chứa nó. Trường hợp xấu nhất là **một vòng dài `n`** (ví dụ `nums[i] = (i+1) % n`), khi đó mỗi `k` mất `n` bước, tổng `O(n²)`.
- **Space:** `O(n)` cho `set` (tạo mới từng `k`, nhưng mỗi lần tối đa `n` phần tử).

Nút thắt: ta **đi lại cùng một vòng nhiều lần** (một lần cho mỗi phần tử của vòng). Hãy đo để thấy thời gian tăng gấp ~4 lần khi `n` gấp đôi.

In [10]:
def one_big_cycle(n):
    return [(i + 1) % n for i in range(n)]

for n in (500, 1000, 2000):
    nums = one_big_cycle(n)
    res, dt = timeit_once(array_nesting_brute, nums)
    print(f"n = {n:5d} -> đáp án {res:5d}, thời gian {dt * 1000:8.1f} ms")
print("Với n = 10^5 sẽ mất cỡ 10^10 bước: không thể nộp được.")

n =   500 -> đáp án   500, thời gian     17.7 ms
n =  1000 -> đáp án  1000, thời gian     67.1 ms


n =  2000 -> đáp án  2000, thời gian    272.4 ms
Với n = 10^5 sẽ mất cỡ 10^10 bước: không thể nộp được.


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** một hoán vị luôn tách thành các **chu trình rời nhau** (disjoint cycles), và `s[k]` chính là chu trình chứa `k`. Vì thế:

- Chu trình nào ta đã đi qua rồi thì **không cần đi lại**: mọi ô của nó đã biết độ dài.
- Chỉ cần đánh dấu mọi ô trên chu trình khi đi qua; sau đó bắt đầu đi tiếp **chỉ từ ô chưa được đánh dấu**.

Mỗi ô được thăm **đúng một lần** trong toàn bộ thuật toán, nên tổng thời gian là `O(n)`.

Tại sao vòng không bao giờ "đi vào" một ô đã thăm của vòng khác? Vì mỗi ô có đúng **một mũi tên vào**: nếu một ô `x` của vòng A có mũi tên vào từ vòng B, thì `x` có hai mũi tên vào (một từ A, một từ B), mâu thuẫn với "giá trị duy nhất". Hãy kiểm chứng trực tiếp: tập các `s[k]` chỉ có thể trùng nhau hoàn toàn hoặc rời nhau hoàn toàn.

In [11]:
def check_cycles_disjoint(nums):
    sets = [frozenset(nesting_list(nums, k)) for k in range(len(nums))]
    distinct = set(sets)
    # Hai tập phân biệt thì phải rời nhau
    lst = list(distinct)
    for a in range(len(lst)):
        for b in range(a + 1, len(lst)):
            assert lst[a].isdisjoint(lst[b])
    return sorted(len(x) for x in distinct)

rng = random.Random(7)
for _ in range(200):
    n = rng.randint(1, 15)
    nums = list(range(n))
    rng.shuffle(nums)
    cyc = check_cycles_disjoint(nums)
    assert sum(cyc) == n          # các chu trình phủ đúng n phần tử, không thừa không thiếu
print("200 hoán vị ngẫu nhiên: các chu trình luôn rời nhau và phủ đúng n phần tử")
print("ví dụ 1 có các chu trình độ dài:", check_cycles_disjoint([5, 4, 0, 3, 1, 6, 2]))

200 hoán vị ngẫu nhiên: các chu trình luôn rời nhau và phủ đúng n phần tử
ví dụ 1 có các chu trình độ dài: [1, 2, 4]


## 7. Các cách tiếp theo

Ngoài Cách 1 (brute force), ta có thêm 4 cách khác nhau thật về ý tưởng (tổng 5 cách):

- **Cách 2:** mảng `visited` riêng (cách chuẩn, dễ hiểu nhất).
- **Cách 3:** đánh dấu **ngay trên `nums`** bằng giá trị đặc biệt, bỏ mảng `visited`.
- **Cách 4:** **Union-Find**: ghép `i` với `nums[i]`; đáp án là kích thước nhóm lớn nhất.
- **Cách 5:** **hoán đổi kiểu cycle sort**: đưa từng phần tử về đúng chỗ, số lần hoán đổi tiết lộ độ dài chu trình.

### 7.1. Cách 2 — Mảng `visited`: mỗi chu trình đi đúng một lần

Giải thích từng đoạn:

1. `visited = [False] * n`, `best = 0`.
2. Với mỗi `i`: nếu `visited[i]` đã `True` thì bỏ qua (ô này thuộc chu trình đã tính).
3. Ngược lại, đây là ô đầu tiên của một chu trình mới. Đi theo mũi tên, mỗi ô đi qua thì đánh dấu `True` và đếm `length += 1`, cho tới khi gặp ô đã đánh dấu (chính là ô xuất phát).
4. `best = max(best, length)`.

Vì ta chỉ đi từ ô chưa đánh dấu và chu trình chứa ô đó **chưa có ô nào đánh dấu** (các chu trình rời nhau), nên vòng `while` dừng đúng khi quay về ô xuất phát.

In [12]:
def array_nesting_visited(nums):
    n = len(nums)
    visited = [False] * n
    best = 0
    for i in range(n):
        if visited[i]:
            continue                      # ô này đã thuộc một chu trình đã tính
        length = 0
        cur = i
        while not visited[cur]:           # đi hết chu trình chứa i
            visited[cur] = True
            cur = nums[cur]
            length += 1
        best = max(best, length)
    return best

for nums, expected in examples:
    got = array_nesting_visited(nums)
    print(nums, "->", got)
    assert got == expected

[5, 4, 0, 3, 1, 6, 2] -> 4
[0, 1, 2] -> 1


Code nộp LeetCode (Cách 2):

In [13]:
class Solution2:
    def arrayNesting(self, nums: List[int]) -> int:
        n = len(nums)
        visited = [False] * n
        best = 0
        for i in range(n):
            if visited[i]:
                continue
            length = 0
            cur = i
            while not visited[cur]:
                visited[cur] = True
                cur = nums[cur]
                length += 1
            best = max(best, length)
        return best

for nums, expected in examples:
    assert Solution2().arrayNesting(nums) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(n)` (mỗi ô thăm đúng một lần, vòng `for` thêm `n` lần kiểm tra); Space `O(n)` cho `visited`.

### 7.2. Cách 3 — Đánh dấu ngay trên `nums` (không cần mảng phụ)

Ý tưởng: mảng đầu vào chứa giá trị `0..n-1`, nên ta có thể dùng một giá trị **ngoài khoảng đó** (ví dụ `-1`) làm dấu "đã thăm". Khi đi qua ô `cur`, ta ghi nhớ ô kế tiếp `nxt = nums[cur]`, rồi đặt `nums[cur] = -1`, và đi tiếp tới `nxt`. Ô nào có `nums[i] == -1` thì đã thăm.

Cái giá phải trả: **hàm làm hỏng mảng đầu vào**. Trên LeetCode điều này thường chấp nhận được (không dùng lại `nums`), nhưng trong code thực tế bạn phải ghi chú hoặc sao chép trước.

In [14]:
def array_nesting_inplace(nums):
    best = 0
    for i in range(len(nums)):
        if nums[i] == -1:
            continue
        length = 0
        cur = i
        while nums[cur] != -1:
            nxt = nums[cur]
            nums[cur] = -1                # đánh dấu đã thăm
            cur = nxt
            length += 1
        best = max(best, length)
    return best

for nums, expected in examples:
    arr = list(nums)                      # sao chép để không làm hỏng ví dụ gốc
    got = array_nesting_inplace(arr)
    print(nums, "->", got, "| mảng sau khi chạy:", arr)
    assert got == expected

[5, 4, 0, 3, 1, 6, 2] -> 4 | mảng sau khi chạy: [-1, -1, -1, -1, -1, -1, -1]
[0, 1, 2] -> 1 | mảng sau khi chạy: [-1, -1, -1]


In [15]:
class Solution3:
    def arrayNesting(self, nums: List[int]) -> int:
        best = 0
        for i in range(len(nums)):
            if nums[i] == -1:
                continue
            length = 0
            cur = i
            while nums[cur] != -1:
                nxt = nums[cur]
                nums[cur] = -1
                cur = nxt
                length += 1
            best = max(best, length)
        return best

for nums, expected in examples:
    assert Solution3().arrayNesting(list(nums)) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n)`; Space `O(1)` bộ nhớ phụ (chỉ vài biến), nhưng **sửa đầu vào**.

### 7.3. Cách 4 — Union-Find (hợp nhất tập rời nhau)

Góc nhìn khác hẳn: coi mỗi chỉ số là một "đỉnh", và mũi tên `i -> nums[i]` là một **cạnh nối** `i` với `nums[i]`. Hai ô thuộc cùng một chu trình khi và chỉ khi chúng nối được với nhau qua các cạnh. Vậy:

- Khởi tạo `n` nhóm, mỗi nhóm một phần tử (`parent[i] = i`, `size[i] = 1`).
- Với mỗi `i`, ghép nhóm của `i` với nhóm của `nums[i]` (`union`).
- Cuối cùng, kích thước nhóm lớn nhất chính là độ dài chu trình dài nhất.

Hai hàm cốt lõi:

- `find(x)`: tìm "đại diện" (gốc) của nhóm chứa `x`, kèm **nén đường** (path compression) để các lần sau nhanh hơn.
- `union(a, b)`: nối gốc nhỏ vào gốc lớn (union by size), và cộng dồn kích thước.

Cách này chậm hơn hằng số so với Cách 2, nhưng rất đáng biết vì nó giải được cả những biến thể khi đồ thị tổng quát hơn hoán vị (ví dụ đếm số thành phần liên thông).

In [16]:
def array_nesting_dsu(nums):
    n = len(nums)
    parent = list(range(n))
    size = [1] * n

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:          # nén đường: trỏ thẳng mọi nút về gốc
            parent[x], x = root, parent[x]
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]

    for i in range(n):
        union(i, nums[i])
    return max(size[find(i)] for i in range(n))

for nums, expected in examples:
    got = array_nesting_dsu(nums)
    print(nums, "->", got)
    assert got == expected

[5, 4, 0, 3, 1, 6, 2] -> 4
[0, 1, 2] -> 1


In [17]:
class Solution4:
    def arrayNesting(self, nums: List[int]) -> int:
        n = len(nums)
        parent = list(range(n))
        size = [1] * n

        def find(x: int) -> int:
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        for i in range(n):
            ra, rb = find(i), find(nums[i])
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]
        return max(size[find(i)] for i in range(n))

for nums, expected in examples:
    assert Solution4().arrayNesting(nums) == expected
print("Solution4 OK")

Solution4 OK


Lưu ý dòng `parent[x], x = root, parent[x]`: vế phải tính xong `(root, parent[x])` trước, rồi gán `parent[x] = root` **trước** `x = ...`. Thứ tự gán từ trái sang phải ở đây là **đúng ý** (gán `parent[x]` khi `x` còn là giá trị cũ). Đây là ví dụ ngược lại với cái bẫy ở mục 3.5, nên hãy luôn kiểm tra thứ tự khi viết kiểu này.

**Độ phức tạp Cách 4:** Time `O(n · α(n))` với `α` là hàm Ackermann ngược (coi như hằng số nhỏ, gần `O(n)`); Space `O(n)` cho `parent` và `size`.

### 7.4. Cách 5 — Hoán đổi kiểu cycle sort

Ý tưởng: ta muốn đưa mỗi giá trị `v` về đúng vị trí `v` (tức biến `nums` thành `[0,1,2,...]`). Với vị trí `i`, trong khi `nums[i] != i`, lấy `j = nums[i]` rồi **hoán đổi** `nums[i]` với `nums[j]`. Sau phép đó, vị trí `j` chứa đúng giá trị `j` (đã về chỗ) và ta tiếp tục với `nums[i]` mới.

Một chu trình độ dài `L` cần đúng **`L - 1` lần hoán đổi** để về chỗ hết (lần cuối cùng đưa 2 phần tử về chỗ cùng lúc). Vậy `độ dài = số lần hoán đổi + 1`. Ô đã đúng chỗ ngay từ đầu cho `0` lần hoán đổi, tức độ dài `1`. Ô thuộc chu trình đã xử lý rồi thì `nums[i] == i` ngay, nên không bị đếm lại.

Nhắc lại cái bẫy mục 3.5: phải lưu `j = nums[i]` trước khi hoán đổi.

In [18]:
def array_nesting_swaps(nums):
    best = 1
    for i in range(len(nums)):
        swaps = 0
        while nums[i] != i:
            j = nums[i]
            nums[i], nums[j] = nums[j], nums[i]
            swaps += 1
        best = max(best, swaps + 1)
    return best

for nums, expected in examples:
    arr = list(nums)
    got = array_nesting_swaps(arr)
    print(nums, "->", got, "| mảng sau khi chạy (đã được sắp xếp):", arr)
    assert got == expected
    assert arr == sorted(nums)

[5, 4, 0, 3, 1, 6, 2] -> 4 | mảng sau khi chạy (đã được sắp xếp): [0, 1, 2, 3, 4, 5, 6]
[0, 1, 2] -> 1 | mảng sau khi chạy (đã được sắp xếp): [0, 1, 2]


In [19]:
class Solution5:
    def arrayNesting(self, nums: List[int]) -> int:
        best = 1
        for i in range(len(nums)):
            swaps = 0
            while nums[i] != i:
                j = nums[i]
                nums[i], nums[j] = nums[j], nums[i]
                swaps += 1
            best = max(best, swaps + 1)
        return best

for nums, expected in examples:
    assert Solution5().arrayNesting(list(nums)) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp Cách 5:** Time `O(n)` (mỗi lần hoán đổi đưa ít nhất một phần tử về đúng chỗ vĩnh viễn, tối đa `n - 1` lần hoán đổi tổng cộng); Space `O(1)` bộ nhớ phụ, **sửa đầu vào** (cuối cùng mảng thành `0..n-1`).

Bảng tóm tắt 5 cách đặt ở mục 16.

## 8. TRACE từng bước

Trace Cách 2 (`visited`) trên ví dụ 1: `nums = [5,4,0,3,1,6,2]`, `n = 7`.

| bước | i | visited[i] trước | hành động | chu trình đi qua | length | best |
|---|---|---|---|---|---|---|
| 1 | 0 | False | đi `0 -> 5 -> 6 -> 2 -> 0` | 0, 5, 6, 2 | 4 | 4 |
| 2 | 1 | False | đi `1 -> 4 -> 1` | 1, 4 | 2 | 4 |
| 3 | 2 | True | bỏ qua | | | 4 |
| 4 | 3 | False | đi `3 -> 3` | 3 | 1 | 4 |
| 5 | 4 | True | bỏ qua | | | 4 |
| 6 | 5 | True | bỏ qua | | | 4 |
| 7 | 6 | True | bỏ qua | | | 4 |

Kết quả `4`. Quan sát: chỉ 3 trong 7 giá trị `i` thật sự bắt đầu một chuyến đi; 4 giá trị còn lại bị bỏ qua nhờ `visited`. Hàm dưới in đúng từng bước.

In [20]:
def trace_nesting(nums):
    n = len(nums)
    visited = [False] * n
    best = 0
    print("nums =", nums)
    for i in range(n):
        if visited[i]:
            print(f"i={i}: đã thăm -> bỏ qua")
            continue
        path = [i]
        length = 0
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
            length += 1
            if not visited[cur]:
                path.append(cur)
        best = max(best, length)
        print(f"i={i}: chu trình {path} + quay về {cur} | length={length} | best={best}")
    print("visited cuối:", visited)
    print("đáp án:", best)
    return best

assert trace_nesting([5, 4, 0, 3, 1, 6, 2]) == 4

nums = [5, 4, 0, 3, 1, 6, 2]
i=0: chu trình [0, 5, 6, 2] + quay về 0 | length=4 | best=4
i=1: chu trình [1, 4] + quay về 1 | length=2 | best=4
i=2: đã thăm -> bỏ qua
i=3: chu trình [3] + quay về 3 | length=1 | best=4
i=4: đã thăm -> bỏ qua
i=5: đã thăm -> bỏ qua
i=6: đã thăm -> bỏ qua
visited cuối: [True, True, True, True, True, True, True]
đáp án: 4


Trace thêm một ví dụ có nhiều chu trình nhỏ: `nums = [1,2,0,4,3,6,5,7]`. Ta kỳ vọng các chu trình `{0,1,2}` (3), `{3,4}` (2), `{5,6}` (2), `{7}` (1) nên đáp án là `3`.

In [21]:
assert trace_nesting([1, 2, 0, 4, 3, 6, 5, 7]) == 3

nums = [1, 2, 0, 4, 3, 6, 5, 7]
i=0: chu trình [0, 1, 2] + quay về 0 | length=3 | best=3
i=1: đã thăm -> bỏ qua
i=2: đã thăm -> bỏ qua
i=3: chu trình [3, 4] + quay về 3 | length=2 | best=3
i=4: đã thăm -> bỏ qua
i=5: chu trình [5, 6] + quay về 5 | length=2 | best=3
i=6: đã thăm -> bỏ qua
i=7: chu trình [7] + quay về 7 | length=1 | best=3
visited cuối: [True, True, True, True, True, True, True, True]
đáp án: 3


Trace Cách 5 (hoán đổi) trên `nums = [5,4,0,3,1,6,2]` để thấy "độ dài = số lần hoán đổi + 1": hàm sau in mảng sau mỗi lần hoán đổi.

In [22]:
def trace_swaps(nums):
    nums = list(nums)
    best = 1
    print("bắt đầu:", nums)
    for i in range(len(nums)):
        swaps = 0
        while nums[i] != i:
            j = nums[i]
            nums[i], nums[j] = nums[j], nums[i]
            swaps += 1
            print(f"  i={i}: đổi nums[{i}] với nums[{j}] -> {nums}")
        print(f"i={i}: swaps={swaps} -> độ dài chu trình = {swaps + 1}")
        best = max(best, swaps + 1)
    print("đáp án:", best)
    return best

assert trace_swaps([5, 4, 0, 3, 1, 6, 2]) == 4

bắt đầu: [5, 4, 0, 3, 1, 6, 2]
  i=0: đổi nums[0] với nums[5] -> [6, 4, 0, 3, 1, 5, 2]
  i=0: đổi nums[0] với nums[6] -> [2, 4, 0, 3, 1, 5, 6]
  i=0: đổi nums[0] với nums[2] -> [0, 4, 2, 3, 1, 5, 6]
i=0: swaps=3 -> độ dài chu trình = 4
  i=1: đổi nums[1] với nums[4] -> [0, 1, 2, 3, 4, 5, 6]
i=1: swaps=1 -> độ dài chu trình = 2
i=2: swaps=0 -> độ dài chu trình = 1
i=3: swaps=0 -> độ dài chu trình = 1
i=4: swaps=0 -> độ dài chu trình = 1
i=5: swaps=0 -> độ dài chu trình = 1
i=6: swaps=0 -> độ dài chu trình = 1
đáp án: 4


## 9. Code nộp LeetCode

Ta chọn **Cách 2 (`visited`)**: dễ hiểu nhất, `O(n)`, không làm hỏng đầu vào.

In [23]:
class Solution:
    def arrayNesting(self, nums: List[int]) -> int:
        n = len(nums)
        visited = [False] * n
        best = 0
        for i in range(n):
            if visited[i]:
                continue
            length = 0
            cur = i
            while not visited[cur]:
                visited[cur] = True
                cur = nums[cur]
                length += 1
            best = max(best, length)
        return best


sol = Solution()
for nums, expected in examples:
    got = sol.arrayNesting(nums)
    print(nums, "->", got)
    assert got == expected

[5, 4, 0, 3, 1, 6, 2] -> 4
[0, 1, 2] -> 1


## 10. Vì sao đúng

**Khẳng định 1 (đi theo mũi tên là một chu trình).** Đặt `f(i) = nums[i]`. Vì `nums` là hoán vị, `f` là **song ánh** trên `{0..n-1}` (mỗi giá trị đúng một lần). Xét dãy `x0 = k, x1 = f(k), x2 = f(f(k)), ...`. Vì chỉ có hữu hạn giá trị, tồn tại `a < b` với `x_a = x_b`. Nếu `a > 0` thì `f(x_{a-1}) = f(x_{b-1})` suy ra `x_{a-1} = x_{b-1}` do `f` đơn ánh; lặp lại ta được `x_0 = x_{b-a}`. Vậy dãy **quay về chính `k`** sau `L = b - a` bước, và `x_0, ..., x_{L-1}` đôi một khác nhau. Tập `{x_0..x_{L-1}}` là **chu trình của `k`**, `L` phần tử.

**Khẳng định 2 (`s[k]` có độ dài đúng bằng `L`).** `s[k]` bắt đầu từ `x_1`, đi qua `x_1, ..., x_L = x_0`, rồi `x_{L+1} = x_1` đã có nên dừng. Vậy `s[k]` gồm đúng `L` phần tử khác nhau (chính chu trình của `k`), `|s[k]| = L`.

**Khẳng định 3 (các chu trình rời nhau hoặc trùng hẳn).** Nếu hai chu trình có chung một ô `y`, thì cả hai đều bằng `{y, f(y), f(f(y)), ...}` nên trùng nhau. Vậy các ô được chia thành các lớp rời nhau; mọi `k` trong cùng một lớp cho cùng độ dài.

**Khẳng định 4 (thuật toán `visited` đúng).** Khi gặp `i` chưa đánh dấu: các chu trình đã đi trước đó bị đánh dấu **hoàn toàn**, nên chu trình của `i` **chưa có ô nào đánh dấu** (nếu có thì đã nằm trọn trong chu trình cũ, tức `i` đã đánh dấu). Vòng `while` đi từ `i`, đánh dấu từng ô, và dừng khi `cur` là ô đã đánh dấu; theo Khẳng định 1 đó chính là `i` sau đúng `L` bước. Vậy `length = L`. Mọi chu trình đều được tính (vì mọi ô `i` hoặc đã đánh dấu hoặc bắt đầu một chu trình mới), nên `best` là độ dài lớn nhất.

**Các cách khác.** Union-Find: các cạnh `(i, nums[i])` nối đúng các ô trong cùng chu trình và không nối chu trình khác, nên thành phần liên thông = chu trình. Cách 3 chính là Cách 2 với dấu `-1` thay cho `visited`. Cách 5: mỗi lần hoán đổi tại `i` đưa `nums[i]` về vị trí `j = nums[i]` vĩnh viễn, đúng `L - 1` lần cho chu trình `L` phần tử (lần cuối đưa 2 phần tử về chỗ cùng lúc).

## 11. Time / Space Complexity

- **Cách 1 (brute):** Time `O(n²)` xấu nhất; Space `O(n)`.
- **Cách 2 (`visited`, nộp):** Time `O(n)`; Space `O(n)`.
- **Cách 3 (đánh dấu trong `nums`):** Time `O(n)`; Space `O(1)` phụ, sửa đầu vào.
- **Cách 4 (Union-Find):** Time `O(n · α(n))`; Space `O(n)`.
- **Cách 5 (hoán đổi):** Time `O(n)`; Space `O(1)` phụ, sửa đầu vào.

Lý do `O(n)` của cách nộp: vòng `for` chạy `n` lần; vòng `while` bên trong chạy tổng cộng **đúng `n` lần** trên toàn bộ thuật toán vì mỗi ô bị đánh dấu `True` tối đa một lần và không bao giờ bị thăm lại. Hãy đo ở `n = 10^5` với 3 loại hoán vị: một vòng lớn, toàn điểm bất động, và ngẫu nhiên.

In [24]:
def one_big_cycle(n):
    return [(i + 1) % n for i in range(n)]

def identity_perm(n):
    return list(range(n))

def random_perm(n, seed):
    r = random.Random(seed)
    p = list(range(n))
    r.shuffle(p)
    return p

N = 10 ** 5
cases = {
    "một vòng dài n": one_big_cycle(N),
    "toàn điểm bất động": identity_perm(N),
    "ngẫu nhiên (seed 1)": random_perm(N, 1),
}
fast_methods = [
    ("visited", lambda a: Solution2().arrayNesting(a)),
    ("in-place", lambda a: Solution3().arrayNesting(list(a))),
    ("union-find", lambda a: Solution4().arrayNesting(a)),
    ("swaps", lambda a: Solution5().arrayNesting(list(a))),
]
for cname, arr in cases.items():
    results = []
    for mname, fn in fast_methods:
        res, dt = timeit_once(fn, arr)
        results.append((mname, res, dt))
    assert len({r[1] for r in results}) == 1
    print(f"{cname:22s} đáp án = {results[0][1]:6d} | " + " | ".join(f"{m}: {dt * 1000:6.1f} ms" for m, _, dt in results))

một vòng dài n         đáp án = 100000 | visited:    7.5 ms | in-place:    8.0 ms | union-find:   45.9 ms | swaps:   13.5 ms


toàn điểm bất động     đáp án =      1 | visited:   13.7 ms | in-place:   11.9 ms | union-find:   46.0 ms | swaps:   11.2 ms


ngẫu nhiên (seed 1)    đáp án =  60275 | visited:   32.7 ms | in-place:   19.4 ms | union-find:  148.2 ms | swaps:   30.1 ms


Cả bốn cách chạy dưới một giây ở `n = 10^5`, và cùng cho một đáp án. Cách brute force với một vòng dài `10^5` sẽ cần cỡ `10^10` bước (hàng chục phút), nên ta không chạy.

## 12. Bộ test

Ta kiểm tra theo bốn lớp: ví dụ của đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu ngẫu nhiên (có seed) với oracle làm đúng từng chữ theo đề.

In [25]:
def oracle(nums):
    """Oracle: dựng s[k] đúng định nghĩa của đề cho từng k (độc lập với mọi cách trên)."""
    best = 0
    for k in range(len(nums)):
        s = []
        cur = nums[k]
        while cur not in s:        # dùng list để không phụ thuộc vào set
            s.append(cur)
            cur = nums[cur]
        best = max(best, len(s))
    return best

all_methods = {
    "brute": array_nesting_brute,
    "visited": lambda a: Solution2().arrayNesting(a),
    "in-place": lambda a: Solution3().arrayNesting(list(a)),
    "union-find": lambda a: Solution4().arrayNesting(a),
    "swaps": lambda a: Solution5().arrayNesting(list(a)),
    "submit": lambda a: Solution().arrayNesting(a),
}

def run_all(nums, expected=None):
    exp = oracle(nums) if expected is None else expected
    for name, fn in all_methods.items():
        got = fn(list(nums))
        assert got == exp, (name, nums, got, exp)
    return exp

# 1) Ví dụ của đề
for nums, expected in examples:
    assert run_all(nums, expected) == expected
print("ví dụ đề: OK")

ví dụ đề: OK


In [26]:
# 2) Edge case
edge = [
    ([0], 1),                                   # n nhỏ nhất
    ([1, 0], 2),                                # một vòng 2 phần tử
    ([0, 1, 2, 3, 4], 1),                       # toàn điểm bất động
    ([1, 2, 3, 4, 0], 5),                       # một vòng dài n
    ([4, 3, 2, 1, 0], 2),                       # đảo ngược: các cặp (0,4), (1,3) và 2 bất động
    ([1, 0, 3, 2, 5, 4], 2),                    # toàn cặp
    ([1, 2, 0, 4, 3, 6, 5, 7], 3),              # nhiều vòng cỡ khác nhau
]
for nums, expected in edge:
    run_all(nums, expected)
    print(nums, "->", expected)
print("edge case: OK")

[0] -> 1
[1, 0] -> 2
[0, 1, 2, 3, 4] -> 1
[1, 2, 3, 4, 0] -> 5
[4, 3, 2, 1, 0] -> 2
[1, 0, 3, 2, 5, 4] -> 2
[1, 2, 0, 4, 3, 6, 5, 7] -> 3
edge case: OK


Phản ví dụ cho vài cách "gần đúng" mà người mới hay nghĩ ra:

- *Chỉ đếm số điểm không bất động* (`nums[i] != i`): với `[1,0,3,2,5,4]` đếm ra `6` nhưng đáp án là `2`.
- *Lấy `max` của `nums`*: sai hoàn toàn vì giá trị không liên quan độ dài.
- *Chỉ thử `k = 0`*: với `[0,2,1]` thử `k=0` ra `1` nhưng đáp án `2`.

In [27]:
def wrong_count_non_fixed(nums):
    return sum(1 for i, v in enumerate(nums) if v != i)

def wrong_only_k0(nums):
    return len(nesting_list(nums, 0))

print("[1,0,3,2,5,4]: đúng =", oracle([1, 0, 3, 2, 5, 4]), "| đếm điểm không bất động =", wrong_count_non_fixed([1, 0, 3, 2, 5, 4]))
print("[0,2,1]      : đúng =", oracle([0, 2, 1]), "| chỉ thử k=0 =", wrong_only_k0([0, 2, 1]))
assert oracle([1, 0, 3, 2, 5, 4]) != wrong_count_non_fixed([1, 0, 3, 2, 5, 4])
assert oracle([0, 2, 1]) != wrong_only_k0([0, 2, 1])

[1,0,3,2,5,4]: đúng = 2 | đếm điểm không bất động = 6
[0,2,1]      : đúng = 2 | chỉ thử k=0 = 1


In [28]:
# 3) Ngẫu nhiên có seed: mọi hoán vị với n nhỏ (vét cạn) + hoán vị ngẫu nhiên n <= 60
from itertools import permutations

cnt = 0
for n in range(1, 7):
    for p in permutations(range(n)):
        run_all(list(p))
        cnt += 1
print("vét cạn mọi hoán vị n <= 6:", cnt, "hoán vị, tất cả các cách khớp oracle")

rng = random.Random(2024)
for t in range(400):
    n = rng.randint(1, 60)
    p = list(range(n))
    rng.shuffle(p)
    run_all(p)
print("400 hoán vị ngẫu nhiên n <= 60: OK")

vét cạn mọi hoán vị n <= 6: 873 hoán vị, tất cả các cách khớp oracle
400 hoán vị ngẫu nhiên n <= 60: OK


In [29]:
# 4) Test lớn: chỉ các cách nhanh (brute bị loại), đáp án biết trước
big = 10 ** 5
cases_big = [
    (one_big_cycle(big), big),
    (identity_perm(big), 1),
    ([big - 1 - i for i in range(big)], 2),      # đảo ngược: các cặp, n chẵn
]
for arr, expected in cases_big:
    for name in ("visited", "in-place", "union-find", "swaps", "submit"):
        assert all_methods[name](arr) == expected, name
print("test n = 10^5: OK")

# Ghép hai chu trình độ dài khác nhau: đáp án là độ dài lớn hơn
a = 70000
arr = [(i + 1) % a for i in range(a)] + [a + (i + 1) % (big - a) for i in range(big - a)]
assert sorted(arr) == list(range(big))
assert Solution().arrayNesting(arr) == a
print("hai chu trình 70000 và 30000: OK")

test n = 10^5: OK
hai chu trình 70000 và 30000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không đánh dấu: đúng nhưng quá chậm (TLE)

Code sai (chỉ "chậm", kết quả vẫn đúng): chạy cho mọi `k` mà không nhớ gì. Đó chính là Cách 1. Với một vòng dài `10^5`, LeetCode báo **Time Limit Exceeded**. Ta chỉ cần thêm `visited` để nhảy qua các chu trình đã tính (Cách 2).

In [30]:
# Mô phỏng "nhìn như đúng" nhưng chậm: đếm số bước tổng cộng đã đi
def count_steps_no_memo(nums):
    steps = 0
    for k in range(len(nums)):
        cur = nums[k]
        steps += 1
        while cur != k:
            cur = nums[cur]
            steps += 1
    return steps

def count_steps_visited(nums):
    visited = [False] * len(nums)
    steps = 0
    for i in range(len(nums)):
        if visited[i]:
            continue
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
            steps += 1
    return steps

for n in (100, 1000):
    nums = one_big_cycle(n)
    print(f"n={n}: không nhớ = {count_steps_no_memo(nums)} bước | có visited = {count_steps_visited(nums)} bước")

n=100: không nhớ = 10000 bước | có visited = 100 bước
n=1000: không nhớ = 1000000 bước | có visited = 1000 bước


### Lỗi 2 — Chỉ đánh dấu điểm xuất phát, không đánh dấu cả chu trình

Code sai:

```python
for i in range(n):
    if visited[i]:
        continue
    visited[i] = True          # chỉ đánh dấu ô i!
    ...đi hết chu trình nhưng không đánh dấu các ô khác...
```

Các ô khác của chu trình vẫn "chưa thăm", nên lát nữa chúng bắt đầu lại cùng chu trình đó, quay về `O(n²)` (lỗi 1). Hãy thấy tận mắt bằng cách đếm số lần bắt đầu một chuyến đi.

In [31]:
def buggy_mark_start_only(nums):
    n = len(nums)
    visited = [False] * n
    best, trips = 0, 0
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True                 # BUG: chỉ đánh dấu ô xuất phát
        trips += 1
        length, cur = 1, nums[i]
        while cur != i:
            cur = nums[cur]
            length += 1
        best = max(best, length)
    return best, trips

nums = one_big_cycle(8)
print("đáp án:", buggy_mark_start_only(nums)[0], "| số chuyến đi:", buggy_mark_start_only(nums)[1], "(lẽ ra chỉ 1 chuyến)")
print("đúng   : visited cả chu trình ->", count_steps_visited(nums), "bước tổng")

đáp án: 8 | số chuyến đi: 8 (lẽ ra chỉ 1 chuyến)
đúng   : visited cả chu trình -> 8 bước tổng


### Lỗi 3 — Hoán đổi sai thứ tự gán

Code sai:

```python
nums[i], nums[nums[i]] = nums[nums[i]], nums[i]
```

Như đã thấy ở mục 3.5, `nums[i]` bị đổi ở lần gán đầu, nên lần gán thứ hai ghi vào **vị trí sai**, làm hỏng hoán vị (mất một giá trị, trùng một giá trị). Code đúng: `j = nums[i]` trước.

In [32]:
def swaps_buggy(nums):
    nums = list(nums)
    best = 1
    for i in range(len(nums)):
        swaps = 0
        guard = 0
        while nums[i] != i and guard < 100:     # guard: tránh lặp vô hạn khi mảng bị hỏng
            nums[i], nums[nums[i]] = nums[nums[i]], nums[i]   # BUG
            swaps += 1
            guard += 1
        best = max(best, swaps + 1)
    return best, nums

nums = [5, 4, 0, 3, 1, 6, 2]
print("code sai :", swaps_buggy(nums), "(đáp án đúng là 4; mảng bị hỏng, có giá trị trùng)")
print("code đúng:", array_nesting_swaps(list(nums)))

code sai : (101, [5, 4, 0, 3, 1, 0, 5]) (đáp án đúng là 4; mảng bị hỏng, có giá trị trùng)
code đúng: 4


### Lỗi 4 — Đếm lệch 1 phần tử

Code sai: khởi tạo `length = 0` rồi dừng ngay khi `cur == i` **trước khi đếm** phần tử cuối, hoặc khởi tạo `length = 1` nhưng vẫn cộng thêm một lần nữa. Kết quả luôn lệch `±1`. Ví dụ phiên bản sai dưới đây bỏ sót ô xuất phát: với `[0,1,2]` trả `0` thay vì `1`.

In [33]:
def off_by_one(nums):
    n = len(nums)
    visited = [False] * n
    best = 0
    for i in range(n):
        if visited[i]:
            continue
        length = 0
        cur = nums[i]
        visited[i] = True
        while cur != i:                 # BUG: ô i không được đếm, và các ô khác được đánh dấu
            visited[cur] = True
            cur = nums[cur]
            length += 1
        best = max(best, length)
    return best

print("[0,1,2]:", off_by_one([0, 1, 2]), "(đúng là 1)")
print("[5,4,0,3,1,6,2]:", off_by_one([5, 4, 0, 3, 1, 6, 2]), "(đúng là 4)")

[0,1,2]: 0 (đúng là 1)
[5,4,0,3,1,6,2]: 3 (đúng là 4)


### Lỗi 5 — Đệ quy (DFS) với chu trình dài sẽ tràn ngăn xếp

Đề thuộc nhóm Depth-First Search, nên nhiều người viết DFS đệ quy. Python mặc định giới hạn độ sâu đệ quy ~1000, mà chu trình có thể dài `10^5` ⇒ `RecursionError`. Hãy dùng vòng `while` (như trên). Ở đây ta thử tái hiện với một giới hạn nhỏ để không làm hỏng kernel.

In [34]:
def nesting_recursive(nums):
    n = len(nums)
    visited = [False] * n
    def dfs(i):
        if visited[i]:
            return 0
        visited[i] = True
        return 1 + dfs(nums[i])
    return max(dfs(i) for i in range(n))

old = sys.getrecursionlimit()
sys.setrecursionlimit(200)
try:
    nesting_recursive(one_big_cycle(5000))
    print("không lỗi?!")
except RecursionError:
    print("RecursionError với chu trình dài 5000 khi giới hạn đệ quy nhỏ")
finally:
    sys.setrecursionlimit(old)
print("với giới hạn mặc định, chu trình 10^5 cũng tràn; vòng while thì không")

RecursionError với chu trình dài 5000 khi giới hạn đệ quy nhỏ
với giới hạn mặc định, chu trình 10^5 cũng tràn; vòng while thì không


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** thấy "mảng là hoán vị của `0..n-1`" cộng với phép "nhảy `i -> nums[i]`" ⇒ nghĩ ngay **chu trình (cycle)** của hoán vị.
- Một hoán vị luôn là **hợp của các chu trình rời nhau**; bài toán "đi theo mũi tên" thường quy về **độ dài/số lượng chu trình**.
- Muốn mỗi phần tử chỉ xử lý một lần: dùng **`visited`** hoặc một **giá trị đánh dấu** (nếu được sửa đầu vào).
- Chỉ cần **ô đại diện của mỗi chu trình**; không đi lại từ mọi ô.
- Cẩn thận hoán đổi: luôn lưu chỉ số vào biến tạm trước (`j = nums[i]`).
- Tránh DFS đệ quy khi độ sâu có thể tới `10^5`; dùng vòng `while`.
- Các bài cùng họ: **41. First Missing Positive**, **287. Find the Duplicate Number**, **442. Find All Duplicates in an Array**, **2360. Longest Cycle in a Graph**, **1306. Jump Game III**.

## 15. Biến thể

### 15.1. Đếm số chu trình

Chỉ cần đếm số lần ta bắt đầu một chuyến đi mới.

In [35]:
def count_cycles(nums):
    n = len(nums)
    visited = [False] * n
    cycles = 0
    for i in range(n):
        if visited[i]:
            continue
        cycles += 1
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
    return cycles

assert count_cycles([5, 4, 0, 3, 1, 6, 2]) == 3
assert count_cycles([0, 1, 2]) == 3
assert count_cycles([1, 2, 3, 4, 0]) == 1
print("số chu trình của ví dụ 1:", count_cycles([5, 4, 0, 3, 1, 6, 2]))

số chu trình của ví dụ 1: 3


### 15.2. Trả về chính chu trình dài nhất (không chỉ độ dài)

Ta lưu các ô đã đi qua vào một list, nếu dài hơn `best` thì giữ lại.

In [36]:
def longest_cycle_itself(nums):
    n = len(nums)
    visited = [False] * n
    best = []
    for i in range(n):
        if visited[i]:
            continue
        cyc = []
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cyc.append(cur)
            cur = nums[cur]
        if len(cyc) > len(best):
            best = cyc
    return best

cyc = longest_cycle_itself([5, 4, 0, 3, 1, 6, 2])
print(cyc)
assert sorted(cyc) == [0, 2, 5, 6]

[0, 5, 6, 2]


### 15.3. Số lần hoán đổi tối thiểu để sắp `nums` thành `[0,1,...,n-1]`

Mỗi chu trình độ dài `L` cần `L - 1` lần hoán đổi, và hoán đổi hai ô khác chu trình chỉ làm **gộp** hai chu trình (không có lợi). Vậy đáp án là **`n - số chu trình`**.

In [37]:
def min_swaps_to_sort(nums):
    return len(nums) - count_cycles(nums)

# Đối chiếu bằng BFS vét cạn trên các hoán vị nhỏ
from collections import deque
def min_swaps_bfs(nums):
    start = tuple(nums)
    target = tuple(range(len(nums)))
    dist = {start: 0}
    dq = deque([start])
    while dq:
        cur = dq.popleft()
        if cur == target:
            return dist[cur]
        for a in range(len(cur)):
            for b in range(a + 1, len(cur)):
                nxt = list(cur)
                nxt[a], nxt[b] = nxt[b], nxt[a]
                nxt = tuple(nxt)
                if nxt not in dist:
                    dist[nxt] = dist[cur] + 1
                    dq.append(nxt)

from itertools import permutations
for n in range(1, 6):
    for p in permutations(range(n)):
        assert min_swaps_to_sort(list(p)) == min_swaps_bfs(p)
print("min swaps = n - số chu trình: khớp BFS với mọi hoán vị n <= 5")

min swaps = n - số chu trình: khớp BFS với mọi hoán vị n <= 5


### 15.4. Bậc (order) của hoán vị

Bậc của hoán vị là số bước `m > 0` nhỏ nhất sao cho áp dụng hoán vị `m` lần thì mọi phần tử về chỗ cũ. Đó là **bội chung nhỏ nhất (LCM)** của các độ dài chu trình.

In [38]:
def cycle_lengths(nums):
    n = len(nums)
    visited = [False] * n
    out = []
    for i in range(n):
        if visited[i]:
            continue
        length, cur = 0, i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
            length += 1
        out.append(length)
    return out

def permutation_order(nums):
    res = 1
    for L in cycle_lengths(nums):
        res = res * L // gcd(res, L)
    return res

def order_bruteforce(nums):
    n = len(nums)
    cur = list(range(n))
    m = 0
    while True:
        cur = [nums[x] for x in cur]
        m += 1
        if cur == list(range(n)):
            return m

for n in range(1, 7):
    for p in permutations(range(n)):
        assert permutation_order(list(p)) == order_bruteforce(list(p))
print("bậc hoán vị của ví dụ 1:", permutation_order([5, 4, 0, 3, 1, 6, 2]), "= lcm(4, 2, 1) =", 4)

bậc hoán vị của ví dụ 1: 4 = lcm(4, 2, 1) = 4


### 15.5. Không còn là hoán vị: đồ thị hàm tổng quát (ý tưởng bài 2360 — Longest Cycle in a Graph)

Nếu giá trị có thể **trùng** (nhiều mũi tên vào cùng một ô) hoặc có ô **không có mũi tên ra** (`-1`), đi theo mũi tên có thể rơi vào ngõ cụt hoặc vào một "đuôi" dẫn tới chu trình. Khi đó ta lưu **thời điểm** lúc ghé từng ô trong chuyến đi hiện tại; nếu gặp lại một ô của chính chuyến đi này thì độ dài chu trình là hiệu thời điểm.

In [39]:
def longest_cycle(edges):
    n = len(edges)
    seen = [False] * n
    best = -1
    for s in range(n):
        if seen[s]:
            continue
        pos = {}
        cur, step = s, 0
        while cur != -1 and not seen[cur]:
            seen[cur] = True
            pos[cur] = step
            step += 1
            cur = edges[cur]
        if cur != -1 and cur in pos:
            best = max(best, step - pos[cur])
    return best

assert longest_cycle([3, 3, 4, 2, 3]) == 3
assert longest_cycle([2, -1, 3, 1]) == -1
print("longest_cycle OK")

longest_cycle OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Dựng `s[k]` bằng `set` cho từng `k` | `O(n²)` | `O(n)` | Chỉ để hiểu đề/làm oracle trên input nhỏ |
| 2. `visited` (nộp) | Mỗi chu trình đi đúng một lần, cờ `visited` | `O(n)` | `O(n)` | Mặc định: dễ hiểu, không sửa đầu vào |
| 3. Đánh dấu trong `nums` | Dùng `-1` làm dấu đã thăm | `O(n)` | `O(1)` phụ | Khi được phép sửa đầu vào, cần tiết kiệm bộ nhớ |
| 4. Union-Find | Ghép `i` với `nums[i]`, lấy nhóm lớn nhất | `O(n·α(n))` | `O(n)` | Khi bài mở rộng sang thành phần liên thông/ghép nhóm động |
| 5. Hoán đổi (cycle sort) | `độ dài = số hoán đổi + 1` | `O(n)` | `O(1)` phụ | Khi bài vốn muốn sắp xếp tại chỗ / đếm hoán đổi |

Bốn cách sau đều tuyến tính; chọn theo ràng buộc về sửa đầu vào và bộ nhớ.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
nums = [5, 4, 0, 3, 1, 6, 2]
print(nums[nums[2]])
print(nesting_list([2, 0, 1, 4, 3], 3))
print(len(nesting_list([2, 0, 1, 4, 3], 0)))
```

In [40]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1,2,0,4,3,6,5,7]`, hãy liệt kê tay mọi chu trình, ghi độ dài của từng chu trình theo thứ tự ô nhỏ nhất của chu trình tăng dần, rồi cho biết đáp án của bài.

In [41]:
# Bài 2: điền kết quả tính tay
hand_lengths = None     # list độ dài các chu trình, theo thứ tự ô nhỏ nhất tăng dần
hand_answer = None      # đáp án của bài

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 2 (mảng `visited`).

In [42]:
def exercise_3_visited(nums: List[int]) -> int:
    n = len(nums)
    # TODO 1: tạo mảng cờ visited độ dài n, toàn False
    best = 0
    # TODO 2: duyệt i từ 0 đến n-1; nếu visited[i] thì continue
    #     TODO 3: đi theo mũi tên từ i, đánh dấu từng ô và đếm length cho tới khi gặp ô đã đánh dấu
    #     TODO 4: cập nhật best = max(best, length)
    return None


# assert exercise_3_visited([5, 4, 0, 3, 1, 6, 2]) == 4
# assert exercise_3_visited([0, 1, 2]) == 1
# assert exercise_3_visited([1, 0]) == 2

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_inplace(nums)` dùng `-1` làm dấu đã thăm, **không** tạo mảng `visited`. Nhớ rằng hàm được phép sửa `nums`.

In [43]:
def exercise_4_inplace(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_inplace([5, 4, 0, 3, 1, 6, 2]) == 4
# assert exercise_4_inplace([0]) == 1
# assert exercise_4_inplace([1, 2, 3, 4, 0]) == 5

### Bài 5 — Mini challenge: số lần hoán đổi tối thiểu (khó)

Cho hoán vị `nums` của `0..n-1`. Mỗi bước bạn được chọn **hai vị trí bất kỳ** và hoán đổi giá trị. Viết `exercise_5_min_swaps(nums)` trả về số bước tối thiểu để biến `nums` thành `[0,1,...,n-1]`.

Gợi ý: nghĩ tới chu trình; mỗi chu trình độ dài `L` đóng góp bao nhiêu?

```python
exercise_5_min_swaps([1, 0, 3, 2]) -> 2
exercise_5_min_swaps([1, 2, 3, 0]) -> 3
exercise_5_min_swaps([0, 1, 2]) -> 0
```

In [44]:
def exercise_5_min_swaps(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_min_swaps([1, 0, 3, 2]) == 2
# assert exercise_5_min_swaps([1, 2, 3, 0]) == 3
# assert exercise_5_min_swaps([0, 1, 2]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `nums[nums[2]]` nghĩa là tra `nums[2]` trước, rồi dùng kết quả làm chỉ số.
- Mức 2: `nums[2] = 0`, rồi `nums[0] = 5`. Với `[2,0,1,4,3]`, `s[3]` bắt đầu từ `nums[3] = 4`.
- Mức 3: dòng 1 là `5`; dòng 2 là `[4, 3]`; dòng 3: `0 -> 2 -> 1 -> 0` nên độ dài `3`.

**Bài 2**
- Mức 1: đi từ ô `0`: `0 -> 1 -> ...`
- Mức 2: các chu trình là `{0,1,2}`, `{3,4}`, `{5,6}`, `{7}`.
- Mức 3: `hand_lengths = [3, 2, 2, 1]`, `hand_answer = 3`.

**Bài 3**
- Mức 1: giống mục 7.1.
- Mức 2: `while not visited[cur]:` đánh dấu `visited[cur] = True`, rồi `cur = nums[cur]`, `length += 1`.
- Mức 3: sau vòng `while` hãy cập nhật `best`, và `return best` sau vòng `for`.

**Bài 4**
- Mức 1: bỏ qua ô đã có `nums[i] == -1`.
- Mức 2: đi tới `cur`: lưu `nxt = nums[cur]`, gán `nums[cur] = -1`, rồi `cur = nxt`.
- Mức 3: dừng khi `nums[cur] == -1`; nhớ đếm `length`.

**Bài 5**
- Mức 1: một chu trình `L` phần tử cần bao nhiêu hoán đổi?
- Mức 2: `L - 1`. Tổng = `sum(L - 1) = n - số chu trình`.
- Mức 3: đếm số chu trình bằng `visited`, trả `n - cycles`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [45]:
nums_b1 = [5, 4, 0, 3, 1, 6, 2]
ans1_line1 = nums_b1[nums_b1[2]]
ans1_line2 = nesting_list([2, 0, 1, 4, 3], 3)
ans1_line3 = len(nesting_list([2, 0, 1, 4, 3], 0))
print(ans1_line1, ans1_line2, ans1_line3)
assert ans1_line1 == 5
assert ans1_line2 == [4, 3]
assert ans1_line3 == 3

5 [4, 3] 3


### Đáp án Bài 2

Các chu trình: `0 -> 1 -> 2 -> 0` (3 ô), `3 -> 4 -> 3` (2 ô), `5 -> 6 -> 5` (2 ô), `7 -> 7` (1 ô). Đáp án là `3`.

In [46]:
ans2_lengths = [3, 2, 2, 1]
ans2_answer = 3
assert cycle_lengths([1, 2, 0, 4, 3, 6, 5, 7]) == ans2_lengths
assert Solution().arrayNesting([1, 2, 0, 4, 3, 6, 5, 7]) == ans2_answer
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [47]:
def ans3_visited(nums: List[int]) -> int:
    n = len(nums)
    visited = [False] * n
    best = 0
    for i in range(n):
        if visited[i]:
            continue
        length = 0
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
            length += 1
        best = max(best, length)
    return best

assert ans3_visited([5, 4, 0, 3, 1, 6, 2]) == 4
assert ans3_visited([0, 1, 2]) == 1
assert ans3_visited([1, 0]) == 2
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [48]:
def ans4_inplace(nums: List[int]) -> int:
    best = 0
    for i in range(len(nums)):
        if nums[i] == -1:
            continue
        length = 0
        cur = i
        while nums[cur] != -1:
            nxt = nums[cur]
            nums[cur] = -1
            cur = nxt
            length += 1
        best = max(best, length)
    return best

assert ans4_inplace([5, 4, 0, 3, 1, 6, 2]) == 4
assert ans4_inplace([0]) == 1
assert ans4_inplace([1, 2, 3, 4, 0]) == 5
rng = random.Random(5)
for _ in range(100):
    n = rng.randint(1, 20)
    p = list(range(n))
    rng.shuffle(p)
    assert ans4_inplace(list(p)) == oracle(p)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Mỗi chu trình `L` phần tử cần `L - 1` hoán đổi; tổng là `n - số chu trình`.

In [49]:
def ans5_min_swaps(nums: List[int]) -> int:
    n = len(nums)
    visited = [False] * n
    cycles = 0
    for i in range(n):
        if visited[i]:
            continue
        cycles += 1
        cur = i
        while not visited[cur]:
            visited[cur] = True
            cur = nums[cur]
    return n - cycles

assert ans5_min_swaps([1, 0, 3, 2]) == 2
assert ans5_min_swaps([1, 2, 3, 0]) == 3
assert ans5_min_swaps([0, 1, 2]) == 0
for n in range(1, 6):
    for p in permutations(range(n)):
        assert ans5_min_swaps(list(p)) == min_swaps_bfs(p)
print("Bài 5 OK (đối chiếu BFS với mọi hoán vị n <= 5)")

Bài 5 OK (đối chiếu BFS với mọi hoán vị n <= 5)


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def arrayNesting(self, nums: List[int]) -> int:
        n = len(nums)
        visited = [False] * n
        best = 0
        for i in range(n):
            if visited[i]:
                continue                  # chu trình này đã tính
            length = 0
            cur = i
            while not visited[cur]:       # đi hết một chu trình
                visited[cur] = True
                cur = nums[cur]
                length += 1
            best = max(best, length)
        return best
```

Nhớ ba ý: **hoán vị = các chu trình rời nhau**, **`s[k]` = chu trình chứa `k`**, **mỗi chu trình chỉ đi một lần**.

### Checklist trước khi submit

- [ ] Hiểu rằng `nums` là hoán vị nên mọi mũi tên đều quay về điểm xuất phát.
- [ ] Có đánh dấu **mọi ô** trên chu trình đã đi (không chỉ ô xuất phát).
- [ ] `length` bắt đầu từ `0` và tăng sau mỗi ô được đánh dấu (không lệch 1).
- [ ] Không dùng đệ quy cho chu trình dài tới `10^5`.
- [ ] Nếu sửa `nums` tại chỗ, đã cân nhắc việc làm hỏng đầu vào (hoặc đã sao chép).
- [ ] Đã thử `[0]`, `[1,0]`, toàn điểm bất động, một vòng dài `n`.
- [ ] Time `O(n)`, Space `O(n)` (hoặc `O(1)` phụ với cách sửa tại chỗ).